El objetivo de este cuaderno es hacer un ensemble entre el mejor modelo de los cuadernos *LLM.ipynb* y *Transformers.ipynb*, junto con un modelo que trabaje con imágenes y otro modelo que trabaje con videos.

En este cuaderno también haremos un análisis de errores.

Dentro de los diferentes tipos de ensembles que podemos hacer nos decantamos por la opción de *Predicción estática* ya que es la opción que se suele hacer en las competiciones dentro del mundo de NLP y en la ciencia de datos porque nos permite ajustes los pesos miles de veces en un segundo sin tener que volver a tirar de tarjeta gráfica para procesar los videos de nuevo.

Los modelos que componen el ensemble son:
- *Mistral* que ha sido el mejor modelo de entre los Transformers y LLM ala hora de evaluar el texto.
- *Convext* entrenando con el dataset de 4 frames como modelo de imágenes.
- *Timesformer* como modelo de video.

Más adelante hemos preparado una celda que calcula cual es la mejor combinación de los pesos de cada modelo al ensemble para obtener los mejores resultados contra el fichero de test estático.

In [1]:
!pip install decord
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 108.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 57.6 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [1]:
import pandas as pd
import numpy as np
import torch
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix
import os
from tqdm import tqdm
from datasets import Dataset, Image

In [2]:
from peft import PeftModel
from PIL import Image as PILImage
from decord import VideoReader, cpu
import decord

ModuleNotFoundError: No module named 'decord'

In [4]:
# Dependencias específicas de modelos
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    pipeline,
    ViTImageProcessor,
    ViTForImageClassification,
    VideoMAEImageProcessor,
    VideoMAEForVideoClassification
)

In [5]:
decord.bridge.set_bridge('torch')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Ensemble Test estático

## 1. Definición de las rutas de los archivos

Definiendo cuáles son las rutas de los archivos .csv sobre los que vamos a hacer el ensemble

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Rutas de datos
CSV_TEST_TEXT = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/EXIST2025_test_3_1.csv"
CSV_IMAGENES = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Frames/Frames-4/dataset_imagenes_test_4_Frames.csv"
RUTA_BASE_VIDEOS = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/videos/"

In [ ]:
# Rutas donde tengo guardados los modelos
DIR_MODELO_TEXTO = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/LLMs/Mistral_7B_QLoRA/checkpoint-565"
DIR_MODELO_IMAGEN = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Vision/CONVNEXT_Frames_4/modelo_final"
DIR_MODELO_VIDEO = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Vision_Temporal/TIMESFORMER_FineTuned_experimentacion/modelo_final"

In [ ]:
# Rutas para guardar las predicciones temporales y resultados
DIR_RESULTADOS = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Ensemble"
os.makedirs(DIR_RESULTADOS, exist_ok=True)
RUTA_PRED_TEXTO = os.path.join(DIR_RESULTADOS, "predicciones_test_mistral.csv")
RUTA_PRED_IMAGEN = os.path.join(DIR_RESULTADOS, "predicciones_test_convnext.csv")
RUTA_PRED_VIDEO = os.path.join(DIR_RESULTADOS, "predicciones_test_timesformer.csv")

## 2. Carga del dataset de test (común para todos)

In [ ]:
print("Cargando el dataset de test fijo...")
test_df = pd.read_csv(CSV_TEST_TEXT)

if "Unnamed: 0" in test_df.columns:
    test_df = test_df.drop(columns=["Unnamed: 0"])
test_df = test_df.rename(columns={"label_task_3_1_merged": "label"})
test_df["label"] = test_df["label"].astype(int)

#test_df

Cargando el dataset de test fijo...


## 3. Fase de generación de predicciones

### 3.1. Predicciones de texto (*Mistral*)

In [ ]:
if not os.path.exists(RUTA_PRED_TEXTO):
    print("\n--- Generando predicciones de Texto (Mistral) ---")

    # 1. Cargar Tokenizador y Modelo Base
    model_id = "mistralai/Mistral-7B-Instruct-v0.3"
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizer.padding_side = "right"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForSequenceClassification.from_pretrained(
        model_id, num_labels=2, device_map="auto", torch_dtype=torch.bfloat16
    )
    base_model.config.pad_token_id = tokenizer.pad_token_id

    # 2. Cargar Pesos QLoRA
    model = PeftModel.from_pretrained(base_model, DIR_MODELO_TEXTO)
    model.eval()

    predicciones_texto = []

    # 3. Inferencia
    for index, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inferencia Mistral"):
        inputs = tokenizer(row["text"], return_tensors="pt", padding="max_length", truncation=True, max_length=128).to(device)
        with torch.no_grad():
            outputs = model(**inputs)
            # Aplicamos Softmax para obtener la probabilidad de clase 1
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1)[0]
            prob_misogino = probs[1].item()

        predicciones_texto.append({
            "id_EXIST": row["id_EXIST"],
            "prob_texto": prob_misogino
        })

    pd.DataFrame(predicciones_texto).to_csv(RUTA_PRED_TEXTO, index=False)

    # Limpiamos memoria
    del model, base_model, tokenizer, inputs, outputs
    torch.cuda.empty_cache()
    print("Predicciones de texto guardadas.")

### 3.2. Predicciones imagen (*ConvNext + Mean-Pooling*)

In [ ]:
from transformers import AutoImageProcessor, AutoModelForImageClassification

if not os.path.exists(RUTA_PRED_IMAGEN):
    print("\n--- Generando predicciones de Imagen (ConvNeXt - Mean-Pooling) ---")

    # Usamos Auto* para que cargue la arquitectura ConvNeXt automáticamente
    processor_img = AutoImageProcessor.from_pretrained(DIR_MODELO_IMAGEN)
    model_img = AutoModelForImageClassification.from_pretrained(DIR_MODELO_IMAGEN).to(device)
    model_img.eval()

    df_imagenes = pd.read_csv(CSV_IMAGENES)
    test_ids = test_df['id_EXIST'].unique()
    test_img_df = df_imagenes[df_imagenes['id_EXIST'].isin(test_ids)].copy()

    predicciones_por_video = {}

    for index, row in tqdm(test_img_df.iterrows(), total=len(test_img_df), desc="Inferencia ConvNeXt (Frames)"):
        id_vid = row['id_EXIST']
        try:
            image = PILImage.open(row['path_imagen']).convert("RGB")
            inputs = processor_img(images=image, return_tensors="pt").to(device)
            with torch.no_grad():
                outputs = model_img(**inputs)
                probs = torch.nn.functional.softmax(outputs.logits, dim=-1)[0]
                prob_misogino = probs[1].item()

            if id_vid not in predicciones_por_video:
                predicciones_por_video[id_vid] = []
            predicciones_por_video[id_vid].append(prob_misogino)
        except Exception as e:
            continue

    # Aplicar Mean-Pooling
    predicciones_img_final = []
    for id_vid, probabilidades in predicciones_por_video.items():
        total_frames = len(probabilidades)
        # Hacemos la media de las probabilidades de los fotogramas del vídeo
        prob_mean = sum(probabilidades) / total_frames if total_frames > 0 else 0.5

        predicciones_img_final.append({
            "id_EXIST": id_vid,
            "prob_imagen": prob_mean
        })

    pd.DataFrame(predicciones_img_final).to_csv(RUTA_PRED_IMAGEN, index=False)

    del model_img, processor_img
    try: del inputs, outputs
    except: pass
    torch.cuda.empty_cache()
    print("✅ Predicciones de imagen (ConvNeXt Mean-Pooling) guardadas.")

### 3.3. Predicciones de vídeo (*TimeSFormer*)

In [ ]:
from transformers import AutoImageProcessor, AutoModelForVideoClassification

if not os.path.exists(RUTA_PRED_VIDEO):
    print("\n--- Generando predicciones de Vídeo (TimeSformer Baseline) ---")

    # Usamos Auto* para que cargue la arquitectura TimeSformer sin problemas
    processor_vid = AutoImageProcessor.from_pretrained(DIR_MODELO_VIDEO)
    model_vid = AutoModelForVideoClassification.from_pretrained(DIR_MODELO_VIDEO).to(device)
    model_vid.eval()

    def sample_frame_indices(clip_len, total_frames):
        if total_frames <= clip_len:
            return np.linspace(0, total_frames - 1, num=clip_len, dtype=int).tolist()
        else:
            return np.linspace(0, total_frames - 1, num=clip_len, dtype=int).tolist()

    predicciones_video = []

    for index, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inferencia TimeSformer"):
        id_vid = row["id_EXIST"]
        # Construir ruta del mp4
        val = str(row.get('path_video', id_vid))
        if not val.endswith(".mp4"): val += ".mp4"
        ruta_video = os.path.join(RUTA_BASE_VIDEOS, val)

        prob_misogino = 0.5 # Valor de incertidumbre por defecto

        try:
            vr = VideoReader(ruta_video, ctx=cpu(0))
            frame_indices = sample_frame_indices(16, len(vr)) # TimeSformer usa 16 frames
            frames = vr.get_batch(frame_indices).numpy()

            inputs = processor_vid(list(frames), return_tensors="pt")
            pixel_values = inputs["pixel_values"].to(device)

            with torch.no_grad():
                outputs = model_vid(pixel_values=pixel_values)
                probs = torch.nn.functional.softmax(outputs.logits, dim=-1)[0]
                prob_misogino = probs[1].item()

        except Exception as e:
            # Si el video falla (corrupto/filtro), se queda en 0.5. Silenciamos el print para no romper la barra.
            pass

        predicciones_video.append({
            "id_EXIST": id_vid,
            "prob_video": prob_misogino # ¡Importante mantener este nombre de columna para el ensemble!
        })

    pd.DataFrame(predicciones_video).to_csv(RUTA_PRED_VIDEO, index=False)

    del model_vid, processor_vid
    try: del inputs, outputs
    except: pass
    torch.cuda.empty_cache()
    print("✅ Predicciones de vídeo (TimeSformer) guardadas.")

## 4. Ensemble multimodal

In [ ]:
print("\n--- Ejecutando Ensemble ---")
df_mistral = pd.read_csv(RUTA_PRED_TEXTO)
df_convnext = pd.read_csv(RUTA_PRED_IMAGEN)
df_timesformer = pd.read_csv(RUTA_PRED_VIDEO)

# Unimos todo usando el 'id_EXIST'
df_ensemble = test_df[['id_EXIST', 'label', 'text']].merge(df_mistral, on="id_EXIST", how="left")
df_ensemble = df_ensemble.merge(df_convnext, on="id_EXIST", how="left")
df_ensemble = df_ensemble.merge(df_timesformer, on="id_EXIST", how="left")


--- Ejecutando Ensemble ---


In [ ]:
# Rellenar posibles NaNs con 0.5 (incertidumbre) en caso de que algún modelo fallara en un id
df_ensemble = df_ensemble.fillna(0.5)

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report

print("\n--- Buscando la mejor combinación de pesos ---")

y_true = df_ensemble['label']
mejor_f1 = 0
mejores_pesos = (0, 0, 0)
mejor_y_pred = []
mejor_prob_final = []

# Probamos combinaciones en saltos de 0.05 (5%)
for w_texto in np.arange(0, 1.05, 0.05):
    for w_imagen in np.arange(0, 1.05 - w_texto, 0.05):
        w_video = round(1.0 - w_texto - w_imagen, 2)

        # Filtro de seguridad por los redondeos de coma flotante
        if w_video < 0 or w_video > 1:
            continue

        # Calculamos la probabilidad final con los pesos actuales
        prob_final = (
            (df_ensemble['prob_texto'] * w_texto) +
            (df_ensemble['prob_imagen'] * w_imagen) +
            (df_ensemble['prob_video'] * w_video)
        )

        # Predicción binaria
        y_pred_actual = (prob_final > 0.5).astype(int)

        # Evaluamos
        f1_actual = f1_score(y_true, y_pred_actual, average='macro')

        # Si mejoramos, guardamos el récord y las predicciones
        if f1_actual > mejor_f1:
            mejor_f1 = f1_actual
            mejores_pesos = (w_texto, w_imagen, w_video)
            mejor_y_pred = y_pred_actual
            mejor_prob_final = prob_final


--- Buscando la mejor combinación de pesos ---


In [ ]:
# ==========================================
# APLICAMOS LOS MEJORES RESULTADOS AL DATAFRAME
# ==========================================
df_ensemble['prob_final'] = mejor_prob_final
df_ensemble['prediccion_ensemble'] = mejor_y_pred
y_pred = mejor_y_pred

w_texto, w_imagen, w_video = mejores_pesos

print(f"\n🏆 ¡Búsqueda completada!")
print(f"Distribución ideal -> Texto: {w_texto:.2f} | Imagen: {w_imagen:.2f} | Vídeo: {w_video:.2f}")


🏆 ¡Búsqueda completada!
Distribución ideal -> Texto: 0.35 | Imagen: 0.55 | Vídeo: 0.10


In [ ]:
# Predicción binaria final
#df_ensemble['prediccion_ensemble'] = (df_ensemble['prob_final'] > 0.5).astype(int)

## 5. Evaluación del ensemble

In [ ]:
print("\n" + "="*50)
print("🏆 RESULTADOS DEL MODELO ENSEMBLE MULTIMODAL (PESOS ÓPTIMOS)")
print("="*50)
print(f"F1-Score (Macro): {f1_score(y_true, y_pred, average='macro'):.4f}")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print("\nMatriz de Confusión:\n", confusion_matrix(y_true, y_pred))
print("\nClassification Report:\n", classification_report(y_true, y_pred, target_names=["No Misógino", "Misógino"]))


🏆 RESULTADOS DEL MODELO ENSEMBLE MULTIMODAL (PESOS ÓPTIMOS)
F1-Score (Macro): 0.7261
Accuracy: 0.7291

Matriz de Confusión:
 [[209  52]
 [ 84 157]]

Classification Report:
               precision    recall  f1-score   support

 No Misógino       0.71      0.80      0.75       261
    Misógino       0.75      0.65      0.70       241

    accuracy                           0.73       502
   macro avg       0.73      0.73      0.73       502
weighted avg       0.73      0.73      0.73       502



In [ ]:
# Guardar las predicciones completas del Ensemble Multimodal
ruta_ensemble_final = os.path.join(DIR_RESULTADOS, "predicciones_ensemble_final.csv")
df_ensemble.to_csv(ruta_ensemble_final, index=False)

print(f"✅ ¡Resultados completos del Ensemble guardados en: {ruta_ensemble_final}!")

✅ ¡Resultados completos del Ensemble guardados en: /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Ensemble/predicciones_ensemble_final.csv!


## 6. Análisis de errores

In [ ]:
print("\n🔍 Generando archivo para el Análisis de Errores...")

# Filtrar donde el Ensemble se equivocó
df_errores = df_ensemble[df_ensemble['label'] != df_ensemble['prediccion_ensemble']].copy()

def clasificar_error(row):
    if row['label'] == 1 and row['prediccion_ensemble'] == 0:
        return "Falso Negativo (No lo detectó)"
    elif row['label'] == 0 and row['prediccion_ensemble'] == 1:
        return "Falso Positivo (Alarma falsa)"

df_errores['tipo_error'] = df_errores.apply(clasificar_error, axis=1)


🔍 Generando archivo para el Análisis de Errores...


In [ ]:
# Ordenamos por la 'confianza' del modelo para ver los errores más graves primero
# Error grave = probabilidad muy lejana a 0.5 (ej. Falso Positivo con 0.99 de probabilidad)
df_errores['severidad_error'] = abs(df_errores['prob_final'] - 0.5)
df_errores = df_errores.sort_values(by=['tipo_error', 'severidad_error'], ascending=[True, False])

# Reordenar columnas para que sea fácil de leer en Excel
columnas_excel = [
    'id_EXIST', 'tipo_error', 'label', 'prediccion_ensemble', 'prob_final',
    'prob_texto', 'prob_imagen', 'prob_video', 'text'
]
df_errores = df_errores[columnas_excel]

In [ ]:
ruta_errores = os.path.join(DIR_RESULTADOS, "casos_para_analisis_errores.xlsx")
df_errores.to_excel(ruta_errores, index=False)

print(f"✅ Se han encontrado {len(df_errores)} errores.")
print(f"📁 Archivo Excel de errores guardado en: {ruta_errores}")
print("¡Abre el Excel para analizar cualitativamente dónde falla tu arquitectura multimodal!")

✅ Se han encontrado 136 errores.
📁 Archivo Excel de errores guardado en: /content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/Ensemble/casos_para_analisis_errores.xlsx
¡Abre el Excel para analizar cualitativamente dónde falla tu arquitectura multimodal!


# Ensemble sobre el test oficial de la competición

In [9]:
import gc

## 1. Rutas y Carga del Test Oficial

In [15]:
# 1. Definición de nuevas rutas para el Test Oficial Ciego
DIR_RESULTADOS = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/SubidasCompeticion/Ensemble"
os.makedirs(DIR_RESULTADOS, exist_ok=True)
RUTA_TEST_OFICIAL = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/test/EXIST2025_test_clean.json"
RUTA_BASE_VIDEOS_OFICIAL = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/test/videos/"

# ⚠️ Ajusta esta ruta a donde tengas el CSV con los frames extraídos del test oficial
CSV_IMAGENES_OFICIAL = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/test/Frames/Frames-4/dataset_imagenes_test_oficial_4_Frames.csv"

# Rutas de salida temporales para no machacar las del valid
RUTA_PRED_OFICIAL_TEXTO = os.path.join(DIR_RESULTADOS, "predicciones_oficial_mistral.csv")
RUTA_PRED_OFICIAL_IMAGEN = os.path.join(DIR_RESULTADOS, "predicciones_oficial_convnext.csv")
RUTA_PRED_OFICIAL_VIDEO = os.path.join(DIR_RESULTADOS, "predicciones_oficial_timesformer.csv")
RUTA_SUBMISSION_JSON = os.path.join(DIR_RESULTADOS, "ensemble_submission_oficial.json")

# Rutas donde tengo guardados los modelos
DIR_MODELO_TEXTO = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/SubidasCompeticion/Mistral_7B_Final"
DIR_MODELO_IMAGEN = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/SubidasCompeticion/CONVNEXT_Frames_4/modelo_final"
DIR_MODELO_VIDEO = "/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/SubidasCompeticion/TIMESFORMER_100porcientoDatos/modelo_final"

In [13]:
# 2. Cargar el Test Oficial
print("Cargando el test oficial limpio...")
df_test_oficial = pd.read_json(RUTA_TEST_OFICIAL, orient='index')

# Formatear el ID correctamente
if "id_EXIST" not in df_test_oficial.columns:
    df_test_oficial = df_test_oficial.reset_index().rename(columns={"index": "id_EXIST"})

print(f"Total de vídeos a procesar: {len(df_test_oficial)}")

Cargando el test oficial limpio...
Total de vídeos a procesar: 674


## 2. Inferencia de Texto (Mistral)

In [16]:
if not os.path.exists(RUTA_PRED_OFICIAL_TEXTO):
    print("\n--- Generando predicciones Oficiales de Texto (Mistral) ---")

    model_id = "mistralai/Mistral-7B-Instruct-v0.3"
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizer.padding_side = "right"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForSequenceClassification.from_pretrained(
        model_id, num_labels=2, device_map="auto", torch_dtype=torch.bfloat16
    )
    base_model.config.pad_token_id = tokenizer.pad_token_id
    model = PeftModel.from_pretrained(base_model, DIR_MODELO_TEXTO)
    model.eval()

    predicciones_texto = []

    for index, row in tqdm(df_test_oficial.iterrows(), total=len(df_test_oficial), desc="Inferencia Oficial Mistral"):
        inputs = tokenizer(str(row["text"]), return_tensors="pt", padding="max_length", truncation=True, max_length=128).to(device)
        with torch.no_grad():
            outputs = model(**inputs)
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1)[0]
            prob_misogino = probs[1].item()

        predicciones_texto.append({
            "id_EXIST": row["id_EXIST"],
            "prob_texto": prob_misogino
        })

    pd.DataFrame(predicciones_texto).to_csv(RUTA_PRED_OFICIAL_TEXTO, index=False)

    # Limpiamos memoria escrupulosamente
    del model, base_model, tokenizer, inputs, outputs
    gc.collect()
    torch.cuda.empty_cache()
    print("✅ Predicciones de texto guardadas y memoria liberada.")
else:
    print("✅ Las predicciones de texto ya existen. Saltando inferencia...")


--- Generando predicciones Oficiales de Texto (Mistral) ---


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] MistralForSequenceClassification LOAD REPORT from: mistralai/Mistral-7B-Instruct-v0.3
Key            | Status     | 
---------------+------------+-
lm_head.weight | UNEXPECTED | 
score.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Inferencia Oficial Mistral: 100%|██████████| 674/674 [03:20<00:00,  3.36it/s]


✅ Predicciones de texto guardadas y memoria liberada.


## 3. Inferencia de Imagen (Convnext) + Mean-Pooling

In [18]:
from transformers import AutoImageProcessor, AutoModelForImageClassification

if not os.path.exists(RUTA_PRED_OFICIAL_IMAGEN):
    print("\n--- Generando predicciones Oficiales de Imagen (ConvNeXt - Mean-Pooling) ---")

    processor_img = AutoImageProcessor.from_pretrained(DIR_MODELO_IMAGEN)
    model_img = AutoModelForImageClassification.from_pretrained(DIR_MODELO_IMAGEN).to(device)
    model_img.eval()

    df_imagenes_oficial = pd.read_csv(CSV_IMAGENES_OFICIAL)
    test_ids = df_test_oficial['id_EXIST'].unique()
    test_img_df = df_imagenes_oficial[df_imagenes_oficial['id_EXIST'].isin(test_ids)].copy()

    predicciones_por_video = {}

    for index, row in tqdm(test_img_df.iterrows(), total=len(test_img_df), desc="Inferencia Oficial ConvNeXt"):
        id_vid = row['id_EXIST']
        try:
            image = PILImage.open(row['path_imagen']).convert("RGB")
            inputs = processor_img(images=image, return_tensors="pt").to(device)
            with torch.no_grad():
                outputs = model_img(**inputs)
                probs = torch.nn.functional.softmax(outputs.logits, dim=-1)[0]
                prob_misogino = probs[1].item()

            if id_vid not in predicciones_por_video:
                predicciones_por_video[id_vid] = []
            predicciones_por_video[id_vid].append(prob_misogino)
        except Exception as e:
            continue

    predicciones_img_final = []
    for id_vid, probabilidades in predicciones_por_video.items():
        total_frames = len(probabilidades)
        prob_mean = sum(probabilidades) / total_frames if total_frames > 0 else 0.5
        predicciones_img_final.append({
            "id_EXIST": id_vid,
            "prob_imagen": prob_mean
        })

    pd.DataFrame(predicciones_img_final).to_csv(RUTA_PRED_OFICIAL_IMAGEN, index=False)

    del model_img, processor_img
    gc.collect()
    torch.cuda.empty_cache()
    print("✅ Predicciones de imagen guardadas y memoria liberada.")
else:
    print("✅ Las predicciones de imagen ya existen. Saltando inferencia...")


--- Generando predicciones Oficiales de Imagen (ConvNeXt - Mean-Pooling) ---


Loading weights:   0%|          | 0/182 [00:00<?, ?it/s]

Inferencia Oficial ConvNeXt: 100%|██████████| 1946/1946 [11:45<00:00,  2.76it/s]


✅ Predicciones de imagen guardadas y memoria liberada.


## 4. Inferencia de Vídeo (TimeSFormer)

In [19]:
from transformers import AutoImageProcessor, AutoModelForVideoClassification

if not os.path.exists(RUTA_PRED_OFICIAL_VIDEO):
    print("\n--- Generando predicciones Oficiales de Vídeo (TimeSFormer) ---")

    processor_vid = AutoImageProcessor.from_pretrained(DIR_MODELO_VIDEO)
    model_vid = AutoModelForVideoClassification.from_pretrained(DIR_MODELO_VIDEO).to(device)
    model_vid.eval()

    def sample_frame_indices(clip_len, total_frames):
        if total_frames <= clip_len:
            return np.linspace(0, total_frames - 1, num=clip_len, dtype=int).tolist()
        else:
            return np.linspace(0, total_frames - 1, num=clip_len, dtype=int).tolist()

    predicciones_video = []

    for index, row in tqdm(df_test_oficial.iterrows(), total=len(df_test_oficial), desc="Inferencia Oficial TimeSFormer"):
        id_vid = row["id_EXIST"]
        val = str(row.get('path_video', id_vid))
        if not val.endswith(".mp4"): val += ".mp4"
        ruta_video = os.path.join(RUTA_BASE_VIDEOS_OFICIAL, val)

        prob_misogino = 0.5
        try:
            vr = VideoReader(ruta_video, ctx=cpu(0))
            frame_indices = sample_frame_indices(16, len(vr))
            frames = vr.get_batch(frame_indices).numpy()

            inputs = processor_vid(list(frames), return_tensors="pt")
            pixel_values = inputs["pixel_values"].to(device)

            with torch.no_grad():
                outputs = model_vid(pixel_values=pixel_values)
                probs = torch.nn.functional.softmax(outputs.logits, dim=-1)[0]
                prob_misogino = probs[1].item()
        except Exception as e:
            pass

        predicciones_video.append({
            "id_EXIST": id_vid,
            "prob_video": prob_misogino
        })

    pd.DataFrame(predicciones_video).to_csv(RUTA_PRED_OFICIAL_VIDEO, index=False)

    del model_vid, processor_vid
    gc.collect()
    torch.cuda.empty_cache()
    print("✅ Predicciones de vídeo guardadas y memoria liberada.")
else:
    print("✅ Las predicciones de vídeo ya existen. Saltando inferencia...")


--- Generando predicciones Oficiales de Vídeo (TimeSFormer) ---


Loading weights:   0%|          | 0/249 [00:01<?, ?it/s]

Inferencia Oficial TimeSFormer: 100%|██████████| 674/674 [00:02<00:00, 273.63it/s]


✅ Predicciones de vídeo guardadas y memoria liberada.


## 5. Unión, Aplicación de Pesos y Exportación a PyEvall

In [20]:
import json

In [21]:
print("\n--- Unificando predicciones y generando JSON de competición ---")

# 1. Cargamos y unimos las 3 predicciones
df_mistral_oficial = pd.read_csv(RUTA_PRED_OFICIAL_TEXTO)
df_convnext_oficial = pd.read_csv(RUTA_PRED_OFICIAL_IMAGEN)
df_timesformer_oficial = pd.read_csv(RUTA_PRED_OFICIAL_VIDEO)

df_ensemble_oficial = df_test_oficial[['id_EXIST']].merge(df_mistral_oficial, on="id_EXIST", how="left")
df_ensemble_oficial = df_ensemble_oficial.merge(df_convnext_oficial, on="id_EXIST", how="left")
df_ensemble_oficial = df_ensemble_oficial.merge(df_timesformer_oficial, on="id_EXIST", how="left")
df_ensemble_oficial = df_ensemble_oficial.fillna(0.5)


--- Unificando predicciones y generando JSON de competición ---


In [22]:
# 2. Aplicamos la receta mágica de pesos calculada en la sección 4 de tu cuaderno
w_texto = 0.35
w_imagen = 0.55
w_video = 0.10

print(f"Usando los pesos óptimos -> Texto: {w_texto} | Imagen: {w_imagen} | Vídeo: {w_video}")

prob_final_oficial = (
    (df_ensemble_oficial['prob_texto'] * w_texto) +
    (df_ensemble_oficial['prob_imagen'] * w_imagen) +
    (df_ensemble_oficial['prob_video'] * w_video)
)

Usando los pesos óptimos -> Texto: 0.35 | Imagen: 0.55 | Vídeo: 0.1


In [23]:
# 3. Decisión Binaria
df_ensemble_oficial['prediccion_ensemble'] = (prob_final_oficial > 0.5).astype(int)

In [24]:
# 4. Formateo a PyEvALL
output_json = []
for index, row in df_ensemble_oficial.iterrows():
    entry = {
        "test_case": "EXIST2025",
        "id": str(row["id_EXIST"]),
        "value": "YES" if int(row["prediccion_ensemble"]) == 1 else "NO"
    }
    output_json.append(entry)

with open(RUTA_SUBMISSION_JSON, "w", encoding="utf-8") as f:
    json.dump(output_json, f, indent=2)

print(f"✅ ¡Completado! Archivo final del Ensemble listo para entregar guardado en:\n{RUTA_SUBMISSION_JSON}")

✅ ¡Completado! Archivo final del Ensemble listo para entregar guardado en:
/content/drive/MyDrive/dataset/EXIST 2025 Videos Dataset/training/Research/SubidasCompeticion/Ensemble/ensemble_submission_oficial.json


In [25]:
print("\nBalance de predicciones enviadas al tribunal:")
df_check = pd.DataFrame(output_json)
print(df_check['value'].value_counts())


Balance de predicciones enviadas al tribunal:
value
NO     404
YES    270
Name: count, dtype: int64
